# Explainable, Uncertainty-Aware Fake News Detection with Fine-Tuned DistilBERT under Domain Shift

**Machine Learning subject project**

This notebook runs the whole project from start to finish:

| # | Section | What it shows |
|---|---------|---------------|
| 1 | Setup | Libraries, settings, GPU check |
| 2 | Load data | Kaggle *Fake and Real News* (ISOT) + COVID-19 fake news test set |
| 3 | Leakage check | Why the raw dataset is "too easy" (Reuters datelines, image credits …) |
| 4 | Cleaning | Removing source artifacts, de-duplication |
| 5 | Baseline | TF-IDF + Logistic Regression on raw vs cleaned text, LIME exposes the shortcut |
| 6 | DistilBERT | Fine-tuning a pretrained transformer |
| 7 | Uncertainty | Softmax vs Temperature Scaling vs Monte Carlo Dropout, ECE, reliability diagrams |
| 8 | Selective prediction | Risk–coverage curve, "UNCERTAIN" flag, error detection |
| 9 | Explainability | LIME word importance + a deletion (faithfulness) test |
| 10 | Domain shift | Train on political news → test on COVID-19 health misinformation |
| 11 | Summary + demo | Final results table, `analyse(text)` function |

**How to run (Google Colab):**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Full run takes roughly **40–60 minutes** on a free T4. For a quick test first, set `quick_run=True` in the settings cell (≈10 min).

All figures and tables are saved to the `results/` folder so you can put them in your report.

## 1. Setup

In [ ]:
!pip -q install lime

In [ ]:
import os, re, time, random, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

CFG = dict(
    model_name     = "distilbert-base-uncased",
    max_len        = 256,     # tokens per article (DistilBERT max is 512)
    batch_size     = 16,
    eval_batch     = 64,
    epochs         = 2,
    lr             = 2e-5,
    weight_decay   = 0.01,
    warmup_ratio   = 0.1,
    mc_passes      = 20,      # forward passes for MC Dropout
    ece_bins       = 15,
    uncertain_rate = 0.05,    # flag the least-confident 5% (threshold tuned on VALIDATION set)
    lime_examples  = 20,      # articles to explain with LIME
    lime_samples   = 500,     # perturbed samples per LIME explanation
    seed           = 42,
    quick_run      = False,   # True = small subsample for a fast test run
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if USE_AMP else "no GPU (training will be very slow)")

OUT = "results"; FIG = f"{OUT}/figures"
os.makedirs(FIG, exist_ok=True)
LABELS = {0: "REAL", 1: "FAKE"}

## 2. Load the datasets

* **Training / in-domain:** Kaggle *Fake and Real News Dataset* (ISOT, University of Victoria). `True.csv` = Reuters articles, `Fake.csv` = articles from sites flagged by PolitiFact/Wikipedia. 2016–2017 US politics & world news.
* **Cross-domain test:** COVID-19 Fake News dataset (Constraint@AAAI-2021 shared task, Patwa et al.), English **test** split: 2,140 social-media posts about COVID-19 labelled `real`/`fake`.

The files are downloaded from public GitHub mirrors. If a link ever stops working, download the files manually (Kaggle: *clmentbisaillon/fake-and-real-news-dataset*) and upload `True.csv`, `Fake.csv` and `english_test_with_labels.csv` to the Colab file panel — the code uses local files first.

**Label convention used everywhere: `0 = REAL`, `1 = FAKE`.**

In [ ]:
ISOT_URL  = "https://raw.githubusercontent.com/laxmimerit/fake-real-news-dataset/main/data/"
COVID_URL = "https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/english_test_with_labels.csv"

def read(name, url):
    return pd.read_csv(name) if os.path.exists(name) else pd.read_csv(url)

true_df = read("True.csv", ISOT_URL + "True.csv"); true_df["label"] = 0
fake_df = read("Fake.csv", ISOT_URL + "Fake.csv"); fake_df["label"] = 1
raw = pd.concat([true_df, fake_df], ignore_index=True)
raw["text"] = raw["text"].fillna("").astype(str)

covid = read("english_test_with_labels.csv", COVID_URL)
covid = covid.rename(columns={"tweet": "text"})
covid["label"] = (covid["label"].str.lower() == "fake").astype(int)

print("ISOT :", raw.shape, raw.label.map(LABELS).value_counts().to_dict())
print("COVID:", covid.shape, covid.label.map(LABELS).value_counts().to_dict())
raw.head(3)

## 3. Leakage check — is the dataset "too easy"?

Before training anything, we check for **artifacts**: text patterns that reveal the *source* of an article rather than whether it is true.
If such patterns exist, a model can reach ~99% accuracy without learning anything about misinformation (this is called **shortcut learning**).

In [ ]:
ARTIFACTS = {
    "'(Reuters)' dateline"     : r"\(Reuters\)",
    "word 'Reuters' anywhere"  : r"(?i)\breuters\b",
    "'Featured image'"         : r"(?i)featured image",
    "'Getty Images'"           : r"(?i)getty images",
    "'via' (image credits)"    : r"\bvia\b",
    "URL / pic.twitter"        : r"http|pic\.twitter",
    "@handle"                  : r"@\w+",
    "curly apostrophe ’"       : r"’",
    "'don t' (stripped apostrophe)": r"\b(?:don|doesn|can|isn|didn) t\b",
}

def artifact_table(df, col):
    rows = []
    for name, pat in ARTIFACTS.items():
        rows.append({"pattern": name,
                     "% of REAL": 100 * df[df.label == 0][col].str.contains(pat, regex=True).mean(),
                     "% of FAKE": 100 * df[df.label == 1][col].str.contains(pat, regex=True).mean()})
    return pd.DataFrame(rows).round(1)

leak_before = artifact_table(raw, "text")
leak_before

In [ ]:
print("subject values by class (this column alone separates the classes perfectly):")
print(pd.crosstab(raw.subject, raw.label.map(LABELS)))
print("\nDuplicate article texts:", raw.text.duplicated().sum())
print("Empty / near-empty texts:", (raw.text.str.split().str.len() < 20).sum())
print("\nExample REAL opening:", raw[raw.label==0].text.iloc[0][:120])
print("Example FAKE ending :", raw[raw.label==1].text.iloc[1][-160:])

**Findings:** ~99% of REAL articles contain "(Reuters)", while almost none of the FAKE ones do. FAKE articles carry image credits ("Featured image via … Getty Images"), links and Twitter handles. Even the apostrophes differ: REAL uses the curly `’`, FAKE had its apostrophes stripped by the scraper (`don t`). The `subject` column is also completely different per class. None of these have anything to do with whether a claim is true.

→ We will **not** use `subject`, `date` or `title` (titles in FAKE are often ALL CAPS), and we clean the article text.

## 4. Cleaning and de-duplication

In [ ]:
CLEAN_RULES = [
    (r"\xa0", " "),
    (r"^.{0,150}?\(Reuters\)\s*[-–—]\s*", ""),                                   # "WASHINGTON (Reuters) - "
    (r"The following statements\s*were posted to the verified Twitter accounts.*?accuracy\.", " "),  # Reuters boilerplate
    (r"\(In .{0,40}? story,.*?\)", " "),                                          # Reuters correction notes
    (r"—\s*[^—\n]{0,80}\(@\w+\)\s*\w+\s+\d{1,2},\s*\d{4}", " "),                  # embedded tweet signature
    (r"(?i)(featured\s+image|featured\s+photo|photo|image|screenshot|screen\s*capture|video)s?\s*(credit)?\s*:?\s*(via|by|courtesy|from)\b[^.]{0,120}", " "),
    (r"(?i)getty\s*images", " "),
    (r"(?i)21st\s*century\s*wire\s*(says)?", " "),
    (r"(?i)\bread more\b\s*:?", " "),
    (r"(?i)\breuters\b", " "),                                                    # any remaining source name
    (r"https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\S+\.com/\S+", " "),           # URLs
    (r"@\w+", " "),                                                               # Twitter handles
    (r"(?i)\[?\(?\b(video|watch|breaking|images?)\b\)?\]?\s*:", " "),             # "WATCH:" style tags
    (r"(?i)\bsubscribe\b.*$", " "),
    (r"[’‘'`´]", " "),                   # apostrophes -> same form for both classes
    (r"[“”″]", '"'),
    (r"\s+", " "),
]
_COMPILED = [(re.compile(p), r) for p, r in CLEAN_RULES]

def clean_text(s):
    s = str(s)
    for pat, rep in _COMPILED:
        s = pat.sub(rep, s)
    return s.strip()

raw["clean"] = raw["text"].map(clean_text)
n0 = len(raw)
data = raw[raw["clean"].str.split().str.len() >= 20].copy()          # drop empty / tiny articles
n1 = len(data)
data = data.drop_duplicates("clean")                                   # remove duplicates
conflict = data.groupby("clean").label.nunique()
data = data[~data["clean"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
print(f"{n0} articles -> {n1} after removing short texts -> {len(data)} after de-duplication")
print(data.label.map(LABELS).value_counts().to_dict())

leak_after = artifact_table(data, "clean")
leak_compare = leak_before.merge(leak_after, on="pattern", suffixes=(" (raw)", " (clean)"))
leak_compare.to_csv(f"{OUT}/leakage_before_after.csv", index=False)
leak_compare

After cleaning, the source artifacts are gone. The remaining gap in the `'don t'` row is no longer a scraping artifact: all apostrophes are now removed the same way in both classes, and FAKE articles simply use more informal contractions ("don't", "isn't"). That is **writing style**, which we keep.

In [ ]:
i = data.index[data.label == 1][5]
print("RAW  :", data.text[i][-300:])
print("CLEAN:", data.clean[i][-300:])
j = data.index[data.label == 0][5]
print("\nRAW  :", data.text[j][:200])
print("CLEAN:", data.clean[j][:200])

### Train / validation / test split (70 / 15 / 15, stratified)

The **validation** set is used for choosing the best epoch, fitting the temperature and choosing the "uncertain" threshold.
The **test** set is touched only for final numbers.

In [ ]:
train_df, tmp = train_test_split(data, test_size=0.30, stratify=data.label, random_state=CFG["seed"])
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=CFG["seed"])

if CFG["quick_run"]:
    train_df = train_df.sample(4000, random_state=CFG["seed"])
    val_df   = val_df.sample(1000, random_state=CFG["seed"])
    test_df  = test_df.sample(1000, random_state=CFG["seed"])
    CFG["mc_passes"], CFG["lime_examples"], CFG["lime_samples"] = 10, 6, 300

for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{n:5s} {len(d):6d}  fake share = {d.label.mean():.2f}")

y_val, y_test = val_df.label.values, test_df.label.values

covid["clean"] = covid["text"].map(clean_text)
covid = covid[covid["clean"].str.split().str.len() >= 3].reset_index(drop=True)
y_cov = covid.label.values
print("COVID test posts after cleaning:", len(covid))

## 5. Baseline: TF-IDF + Logistic Regression

A classical baseline tells us how much the transformer actually adds. We train it twice — on **raw** text and on **cleaned** text — to measure the effect of the leakage.

In [ ]:
def metrics(y, prob_fake, name=""):
    pred = (prob_fake >= 0.5).astype(int)
    out = {"model": name,
           "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
           "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0)}
    out["roc_auc"] = roc_auc_score(y, prob_fake) if len(np.unique(y)) > 1 else np.nan
    return out

def make_baseline():
    return make_pipeline(TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=4))

t0 = time.time()
lr_raw   = make_baseline().fit(train_df.text,  train_df.label)
lr_clean = make_baseline().fit(train_df.clean, train_df.label)
print(f"trained in {time.time()-t0:.0f}s")

baseline_rows = [
    metrics(y_test, lr_raw.predict_proba(test_df.text)[:, 1],   "TF-IDF+LR (raw text)"),
    metrics(y_test, lr_clean.predict_proba(test_df.clean)[:, 1], "TF-IDF+LR (clean text)"),
]
pd.DataFrame(baseline_rows).round(4)

In [ ]:
def top_words(pipe, k=15):
    vec, clf = pipe.named_steps["tfidfvectorizer"], pipe.named_steps["logisticregression"]
    names, coef = np.array(vec.get_feature_names_out()), clf.coef_[0]
    return pd.DataFrame({"→ REAL (raw)": names[np.argsort(coef)[:k]], "→ FAKE (raw)": names[np.argsort(-coef)[:k]]})

top_words(lr_raw)

### LIME exposes the shortcut

LIME (Local Interpretable Model-agnostic Explanations) removes words at random, watches how the prediction changes, and fits a small linear model to find which words mattered most for **this one** prediction.
On the raw-text model, the top words are source artifacts ("Reuters", "via", "image") rather than content — explainability is being used here to **debug the dataset**.

In [ ]:
explainer = LimeTextExplainer(class_names=["REAL", "FAKE"], random_state=CFG["seed"])

def shorten(t, n_words=180):          # the transformer only reads ~256 tokens anyway
    return " ".join(str(t).split()[:n_words])

k_real = test_df.index[test_df.label == 0][0]
k_fake = test_df.index[(test_df.label == 1) & test_df.text.str.contains("via")][0]
for k, tag in [(k_real, "real"), (k_fake, "fake")]:
    exp = explainer.explain_instance(shorten(test_df.text[k], 250), lr_raw.predict_proba, num_features=10, num_samples=1000)
    fig = exp.as_pyplot_figure(); plt.title(f"LIME — TF-IDF+LR on RAW text ({tag} article)")
    plt.tight_layout(); plt.savefig(f"{FIG}/lime_raw_baseline_{tag}.png", dpi=150); plt.show()
    print([(str(w), round(float(v), 3)) for w, v in exp.as_list()])

## 6. Fine-tuning DistilBERT

**DistilBERT** is a smaller, faster version of BERT (6 transformer layers instead of 12, ~66M parameters, ~97% of BERT's accuracy). It was pre-trained on English Wikipedia + BooksCorpus.
We add a classification head on top of the `[CLS]` token and fine-tune **all** weights on our cleaned articles (transfer learning).

Settings: max 256 tokens, batch 16, learning rate 2e-5 with linear warm-up/decay, AdamW, 2 epochs, mixed precision on GPU.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"], num_labels=2, id2label=LABELS, label2id={v: k for k, v in LABELS.items()}).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels)
    return enc

train_loader = DataLoader(list(zip(train_df.clean, train_df.label)), batch_size=CFG["batch_size"],
                          shuffle=True, collate_fn=collate)

In [ ]:
@torch.no_grad()
def predict_logits(texts, mc_dropout=False, passes=1, batch_size=None):
    # Returns logits of shape (passes, N, 2). With mc_dropout=True, dropout stays ON at inference.
    batch_size = batch_size or CFG["eval_batch"]
    model.eval()
    if mc_dropout:
        for m in model.modules():
            if isinstance(m, torch.nn.Dropout):
                m.train()                     # only dropout layers switched to "train" mode
    texts = list(texts)
    out = np.zeros((passes, len(texts), 2), dtype=np.float32)
    for s in range(0, len(texts), batch_size):
        enc = tokenizer(texts[s:s + batch_size], truncation=True, max_length=CFG["max_len"],
                        padding=True, return_tensors="pt").to(DEVICE)
        for p in range(passes):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out[p, s:s + batch_size] = model(**enc).logits.float().cpu().numpy()
    model.eval()
    return out

def softmax_np(z, T=1.0):
    return torch.softmax(torch.tensor(z) / T, dim=-1).numpy()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
total_steps = len(train_loader) * CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG["warmup_ratio"] * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history, best_f1, best_state = [], -1, None
for epoch in range(1, CFG["epochs"] + 1):
    model.train(); t0 = time.time(); running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)}  loss {running/step:.4f}")
    val_prob = softmax_np(predict_logits(val_df.clean)[0])[:, 1]
    m = metrics(y_val, val_prob)
    history.append({"epoch": epoch, "train_loss": running / len(train_loader), "val_acc": m["accuracy"], "val_f1": m["f1"]})
    print(f"epoch {epoch}: train loss {running/len(train_loader):.4f} | val acc {m['accuracy']:.4f} | val F1 {m['f1']:.4f} | {time.time()-t0:.0f}s")
    if m["f1"] > best_f1:
        best_f1, best_state = m["f1"], copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})

model.load_state_dict(best_state)
model.save_pretrained(f"{OUT}/distilbert_fakenews"); tokenizer.save_pretrained(f"{OUT}/distilbert_fakenews")
pd.DataFrame(history)

## 7. Uncertainty and calibration

We compare three ways of getting a confidence score:

1. **Softmax** — the raw probability from one normal forward pass. Neural nets are usually **over-confident**.
2. **Temperature scaling** — divide the logits by one number *T* (fitted on the validation set by minimising negative log-likelihood). *T* > 1 softens over-confident predictions. It never changes the predicted class.
3. **Monte Carlo (MC) Dropout** — keep dropout switched **on** at test time and run *T* = 20 forward passes. Each pass uses a slightly different "thinned" network, so we get a *distribution* of predictions (Gal & Ghahramani, 2016, showed this approximates a Bayesian neural network).

From the MC passes we compute:
* **mean probability** p̄ = average of the 20 softmax outputs → final prediction
* **predictive entropy** H[p̄] → *total* uncertainty
* **mutual information (BALD)** = H[p̄] − mean(H[pₜ]) → *epistemic* (model) uncertainty: high when the passes **disagree**. This is the part that should grow on unfamiliar data.

**Expected Calibration Error (ECE):** group predictions into 15 confidence bins; ECE = Σ (bin size / N) × |accuracy(bin) − confidence(bin)|. 0 = perfectly calibrated.

In [ ]:
def entropy(p):
    return -(p * np.log(np.clip(p, 1e-12, 1))).sum(-1)

def mc_stats(logits_passes):
    probs = softmax_np(logits_passes)              # (T, N, 2)
    mean_p = probs.mean(0)
    H = entropy(mean_p)
    MI = H - entropy(probs).mean(0)
    return {"prob": mean_p, "conf": mean_p.max(1), "entropy": H, "mi": MI, "std_fake": probs[..., 1].std(0)}

def ece_score(y, prob, n_bins=CFG["ece_bins"]):
    conf, pred = prob.max(1), prob.argmax(1)
    correct = (pred == y).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)       # binary task: confidence is always >= 0.5
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0.5 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return ece

def full_metrics(y, prob, name):
    m = metrics(y, prob[:, 1], name)
    m["ECE"] = ece_score(y, prob)
    m["NLL"] = -np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean()
    m["Brier"] = ((prob[:, 1] - y) ** 2).mean()
    return m

def fit_temperature(logits, y):
    logT = torch.zeros(1, requires_grad=True)
    L, Y = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([logT], lr=0.05, max_iter=500)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(L / logT.exp(), Y); loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp())

In [ ]:
t0 = time.time()
val_logits  = predict_logits(val_df.clean)[0]
test_logits = predict_logits(test_df.clean)[0]
T_opt = fit_temperature(val_logits, y_val)
print(f"fitted temperature T = {T_opt:.3f}")

print(f"running MC Dropout with {CFG['mc_passes']} passes ...")
mc_val  = mc_stats(predict_logits(val_df.clean,  mc_dropout=True, passes=CFG["mc_passes"]))
mc_test = mc_stats(predict_logits(test_df.clean, mc_dropout=True, passes=CFG["mc_passes"]))
print(f"done in {time.time()-t0:.0f}s")

P = {  # probability matrices for the TEST set, per confidence method
    "Softmax":             softmax_np(test_logits),
    "Temperature scaling": softmax_np(test_logits, T_opt),
    "MC Dropout":          mc_test["prob"],
}
P_val = {"Softmax": softmax_np(val_logits), "Temperature scaling": softmax_np(val_logits, T_opt), "MC Dropout": mc_val["prob"]}

in_domain = pd.DataFrame([full_metrics(y_test, p, f"DistilBERT + {k}") for k, p in P.items()])
in_domain.round(4)

In [ ]:
def reliability(ax, y, prob, title, n_bins=10):
    conf, correct = prob.max(1), (prob.argmax(1) == y)
    bins = np.linspace(0.5, 1.0, n_bins + 1); centers, accs, counts = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf <= hi) if lo == 0.5 else (conf > lo) & (conf <= hi)
        if m.any():
            centers.append(conf[m].mean()); accs.append(correct[m].mean()); counts.append(m.sum())
    ax.plot([0.5, 1], [0.5, 1], "--", color="gray", label="perfect calibration")
    ax.plot(centers, accs, "o-", label="model")
    for c, a, n in zip(centers, accs, counts):
        ax.annotate(str(n), (c, a), textcoords="offset points", xytext=(0, 6), fontsize=7, ha="center")
    ax.set_title(f"{title}\nECE = {ece_score(y, prob):.4f}"); ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_xlim(0.5, 1.01); ax.set_ylim(0.0, 1.02); ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (k, p) in zip(axes, P.items()):
    reliability(ax, y_test, p, k)
plt.suptitle("Reliability diagrams — in-domain test set (numbers = articles per bin)")
plt.tight_layout(); plt.savefig(f"{FIG}/reliability_in_domain.png", dpi=150); plt.show()

## 8. Selective prediction — does low confidence actually mean "likely wrong"?

This is the **confidence–accuracy analysis** (in the literature: *selective prediction* / *classification with a reject option*).

* **Risk–coverage curve:** sort articles from most to least confident. At coverage *c* we keep the top *c*% and measure the error rate (risk) on them. A good confidence score gives a curve that stays low until high coverage.
* **AURC** = area under that curve (lower = better).
* **Error-detection AUROC:** treat "is this prediction wrong?" as a yes/no problem and use the uncertainty as the score. 0.5 = useless, 1.0 = perfect.
* **The "UNCERTAIN" flag:** we choose a confidence threshold τ on the **validation** set so that 5% of validation articles are flagged, then apply that fixed τ to the test set.

In [ ]:
def risk_coverage(y, prob, score=None):
    # score = certainty (higher = more certain). Default = max probability.
    score = prob.max(1) if score is None else score
    order = np.argsort(-score)
    errors = (prob.argmax(1) != y)[order].astype(float)
    k = np.arange(1, len(y) + 1)
    risk = np.cumsum(errors) / k
    return k / len(y), risk, risk.mean()

def error_auroc(y, prob, uncertainty):
    wrong = (prob.argmax(1) != y).astype(int)
    return roc_auc_score(wrong, uncertainty) if 0 < wrong.sum() < len(wrong) else np.nan

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sel_rows = []
for k, p in P.items():
    cov, risk, aurc = risk_coverage(y_test, p)
    ax.plot(cov, risk, label=f"{k} (AURC={aurc:.4f})")
    sel_rows.append({"method": k, "AURC": aurc, "error-detection AUROC": error_auroc(y_test, p, 1 - p.max(1))})
# MC dropout with mutual information as the uncertainty score
cov, risk, aurc = risk_coverage(y_test, P["MC Dropout"], score=-mc_test["mi"])
ax.plot(cov, risk, ":", label=f"MC Dropout, MI score (AURC={aurc:.4f})")
sel_rows.append({"method": "MC Dropout (mutual information)", "AURC": aurc,
                 "error-detection AUROC": error_auroc(y_test, P["MC Dropout"], mc_test["mi"])})
ax.axhline((P["Softmax"].argmax(1) != y_test).mean(), color="gray", ls="--", lw=1, label="no rejection")
ax.set_xlabel("coverage (fraction of articles the model answers)"); ax.set_ylabel("error rate on answered articles")
ax.set_title("Risk–coverage curve (in-domain test)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG}/risk_coverage_in_domain.png", dpi=150); plt.show()
selective = pd.DataFrame(sel_rows); selective.round(4)

In [ ]:
# Threshold tau chosen on VALIDATION (never on test), using MC Dropout confidence
tau = np.quantile(mc_val["conf"], CFG["uncertain_rate"])
print(f"tau (MC-Dropout confidence threshold) = {tau:.4f}")

def three_way(prob, conf, tau):
    return np.where(conf < tau, "UNCERTAIN", np.where(prob[:, 1] >= 0.5, "FAKE", "REAL"))

def flag_report(y, prob, conf, tau, name):
    flagged = conf < tau
    wrong = prob.argmax(1) != y
    return {"set": name, "articles": len(y), "flagged UNCERTAIN %": 100 * flagged.mean(),
            "accuracy (all)": (~wrong).mean(),
            "accuracy (confident only)": (~wrong[~flagged]).mean() if (~flagged).any() else np.nan,
            "accuracy (flagged only)": (~wrong[flagged]).mean() if flagged.any() else np.nan,
            "% of all errors caught by flag": 100 * (wrong & flagged).sum() / max(wrong.sum(), 1)}

flag_rows = [flag_report(y_test, mc_test["prob"], mc_test["conf"], tau, "ISOT test (in-domain)")]
pd.DataFrame(flag_rows).round(4)

## 9. Explainability with LIME (DistilBERT)

For each chosen article, LIME creates `lime_samples` perturbed copies with random words removed, gets DistilBERT's prediction for each, and fits a weighted linear model. The weights = importance of each word **for this prediction**. Dropout is **off** here so the explanation is deterministic.

We explain a mix of: the most uncertain articles, misclassified articles, and random correct ones.
Articles are cut to their first 180 words because the model only reads the first 256 tokens anyway.

In [ ]:
def bert_proba(texts):
    return softmax_np(predict_logits(texts)[0], T_opt)

test_pos = np.arange(len(test_df))
pred_test = P["MC Dropout"].argmax(1)
n = CFG["lime_examples"]
most_uncertain = test_pos[np.argsort(mc_test["conf"])[: n // 4]]
errors_idx     = test_pos[pred_test != y_test][: n // 4]
rng = np.random.default_rng(CFG["seed"])
chosen = list(dict.fromkeys(list(most_uncertain) + list(errors_idx)))
rest = [i for i in rng.permutation(test_pos) if i not in chosen]
chosen = (chosen + rest)[:n]

lime_results = []
t0 = time.time()
for i in chosen:
    text = shorten(test_df.clean.iloc[i])
    exp = explainer.explain_instance(text, bert_proba, num_features=10, num_samples=CFG["lime_samples"])
    p = bert_proba([text])[0]
    lime_results.append({"pos": i, "text": text, "true": LABELS[y_test[i]], "pred": LABELS[int(p.argmax())],
                         "p_fake": p[1], "mc_conf": mc_test["conf"][i], "weights": exp.as_list(), "exp": exp})
print(f"{len(lime_results)} explanations in {time.time()-t0:.0f}s")

pd.DataFrame([{k: r[k] for k in ["true", "pred", "p_fake", "mc_conf"]} |
              {"top words (+ = towards FAKE)": ", ".join(f"{w}({v:+.2f})" for w, v in r["weights"][:5])}
              for r in lime_results]).round(3)

In [ ]:
# Show two explanations in full (one FAKE-predicted, one REAL-predicted) and save them as figures
shown = []
for want in ["FAKE", "REAL"]:
    r = next((r for r in lime_results if r["pred"] == want), None)
    if r is None: continue
    fig = r["exp"].as_pyplot_figure(); plt.title(f"LIME — DistilBERT (true={r['true']}, pred={r['pred']}, P(fake)={r['p_fake']:.2f})")
    plt.tight_layout(); plt.savefig(f"{FIG}/lime_distilbert_{want.lower()}.png", dpi=150); plt.show()
    display(HTML(r["exp"].as_html(text=True)))   # interactive highlighted text

In [ ]:
# Aggregate: which words push predictions towards FAKE / REAL most often across all explained articles?
agg = defaultdict(float)
for r in lime_results:
    for w, v in r["weights"]:
        agg[w.lower()] += v
agg = pd.Series(agg).sort_values()
top = pd.concat([agg.head(12), agg.tail(12)])
plt.figure(figsize=(7, 6))
plt.barh(top.index, top.values, color=["tab:blue" if v < 0 else "tab:red" for v in top.values])
plt.axvline(0, color="k", lw=0.8); plt.xlabel("summed LIME weight  (← REAL | FAKE →)")
plt.title("Most influential words across explained articles")
plt.tight_layout(); plt.savefig(f"{FIG}/lime_aggregate_words.png", dpi=150); plt.show()

### Are the explanations faithful? (deletion test)

A pretty highlight is not proof. If LIME's top words really drive the prediction, **deleting them** should hurt the model's confidence in its predicted class much more than deleting the same number of **random** words.

In [ ]:
def delete_words(text, words):
    for w in words:
        text = re.sub(r"(?<!\w)" + re.escape(w) + r"(?!\w)", " ", text)
    return re.sub(r"\s+", " ", text)

K = 5
rows = []
for r in lime_results:
    base = bert_proba([r["text"]])[0]; c = int(base.argmax())
    sign = 1 if c == 1 else -1                       # LIME weights are towards FAKE
    support = [w for w, v in r["weights"] if sign * v > 0][:K]
    vocab = list(set(re.findall(r"\w+", r["text"])) - set(support))
    rand = list(rng.choice(vocab, size=min(len(support), len(vocab)), replace=False)) if support else []
    p_top  = bert_proba([delete_words(r["text"], support)])[0][c]
    p_rand = bert_proba([delete_words(r["text"], rand)])[0][c]
    rows.append({"drop (LIME top words)": base[c] - p_top, "drop (random words)": base[c] - p_rand})
faith = pd.DataFrame(rows)
print(f"Average fall in predicted-class probability after deleting {K} words:")
faith.mean().round(4)

## 10. Domain shift: political news → COVID-19 health misinformation

The model was trained on long 2016–17 political news articles. The COVID set is **different in topic** (health), **in time** (2020) and **in format** (short social-media posts). A model that truly understood misinformation would transfer; a model that learned the *writing style* of Reuters vs partisan blogs will not.

What we expect and check:
* accuracy / F1 drop compared with in-domain
* calibration gets worse (ECE up) — the model is confidently wrong
* uncertainty (especially **mutual information**) should be **higher** on COVID posts — if it is, the uncertainty flag works as a warning sign for unfamiliar data

In [ ]:
t0 = time.time()
cov_logits = predict_logits(covid.clean)[0]
mc_cov = mc_stats(predict_logits(covid.clean, mc_dropout=True, passes=CFG["mc_passes"]))
print(f"done in {time.time()-t0:.0f}s")

P_cov = {"Softmax": softmax_np(cov_logits), "Temperature scaling": softmax_np(cov_logits, T_opt), "MC Dropout": mc_cov["prob"]}
lr_cov_prob = lr_clean.predict_proba(covid.clean)[:, 1]

rows = []
for dom, y, lrp, PP in [("in-domain (ISOT test)", y_test, lr_clean.predict_proba(test_df.clean)[:, 1], P),
                        ("cross-domain (COVID-19)", y_cov, lr_cov_prob, P_cov)]:
    r = metrics(y, lrp, "TF-IDF+LR (clean)"); r["domain"] = dom; rows.append(r)
    for k, p in PP.items():
        r = full_metrics(y, p, f"DistilBERT + {k}"); r["domain"] = dom; rows.append(r)
domain_table = pd.DataFrame(rows)[["domain", "model", "accuracy", "precision", "recall", "f1", "roc_auc", "ECE"]]
domain_table.round(4)

In [ ]:
print("Confusion matrix on COVID-19 (rows = true REAL/FAKE, cols = predicted REAL/FAKE):")
print(confusion_matrix(y_cov, P_cov["MC Dropout"].argmax(1)))
print(f"\nShare of COVID posts predicted FAKE: {P_cov['MC Dropout'].argmax(1).mean():.2%}  (true share: {y_cov.mean():.2%})")

ind = domain_table[domain_table.domain.str.startswith("in")].set_index("model")
crs = domain_table[domain_table.domain.str.startswith("cross")].set_index("model")
drop = pd.DataFrame({"in-domain accuracy": ind.accuracy, "COVID-19 accuracy": crs.accuracy,
                     "accuracy drop (points)": 100 * (ind.accuracy - crs.accuracy),
                     "in-domain F1": ind.f1, "COVID-19 F1": crs.f1})
drop.round(4)

In [ ]:
flag_rows.append(flag_report(y_cov, mc_cov["prob"], mc_cov["conf"], tau, "COVID-19 (cross-domain)"))
flags = pd.DataFrame(flag_rows); display(flags.round(4))

# Can uncertainty tell in-domain from out-of-domain inputs?
is_ood = np.r_[np.zeros(len(y_test)), np.ones(len(y_cov))]
ood = pd.DataFrame([
    {"uncertainty score": "1 − softmax confidence", "OOD-detection AUROC": roc_auc_score(is_ood, np.r_[1 - P["Softmax"].max(1), 1 - P_cov["Softmax"].max(1)])},
    {"uncertainty score": "MC predictive entropy",  "OOD-detection AUROC": roc_auc_score(is_ood, np.r_[mc_test["entropy"], mc_cov["entropy"]])},
    {"uncertainty score": "MC mutual information",  "OOD-detection AUROC": roc_auc_score(is_ood, np.r_[mc_test["mi"], mc_cov["mi"]])},
])
display(ood.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(np.log10(1 - mc_test["conf"] + 1e-6), bins=30, alpha=.6, density=True, label="ISOT test")
axes[0].hist(np.log10(1 - mc_cov["conf"] + 1e-6),  bins=30, alpha=.6, density=True, label="COVID-19")
axes[0].axvline(np.log10(1 - tau + 1e-6), color="k", ls="--", label="τ (UNCERTAIN to the right)")
axes[0].set_title("log10(1 − MC-Dropout confidence)"); axes[0].legend()
axes[1].hist(np.log10(mc_test["mi"] + 1e-8), bins=30, alpha=.6, density=True, label="ISOT test")
axes[1].hist(np.log10(mc_cov["mi"] + 1e-8),  bins=30, alpha=.6, density=True, label="COVID-19")
axes[1].set_title("Mutual information (log10) — epistemic uncertainty"); axes[1].legend()
reliability(axes[2], y_cov, P_cov["MC Dropout"], "Reliability on COVID-19 (MC Dropout)")
plt.tight_layout(); plt.savefig(f"{FIG}/domain_shift_uncertainty.png", dpi=150); plt.show()

## 11. Summary and demo

In [ ]:
summary = {
    "in-domain metrics":    in_domain,
    "baselines (raw vs clean)": pd.DataFrame(baseline_rows),
    "selective prediction": selective,
    "uncertain flag":       flags,
    "domain shift":         domain_table,
    "accuracy drop":        drop.reset_index(),
    "OOD detection":        ood,
    "LIME deletion test":   faith.mean().to_frame("mean probability drop").reset_index(),
}
with pd.ExcelWriter(f"{OUT}/all_results.xlsx") as xw:
    for name, df_ in summary.items():
        df_.round(4).to_excel(xw, sheet_name=name[:31], index=False)
for name, df_ in summary.items():
    df_.round(4).to_csv(f"{OUT}/{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv", index=False)
print(f"Temperature T = {T_opt:.3f} | uncertainty threshold tau = {tau:.4f}")
print("Saved:", sorted(os.listdir(OUT)), "\nFigures:", sorted(os.listdir(FIG)))

In [ ]:
def analyse(text, passes=None, lime_samples=300):
    passes = passes or CFG["mc_passes"]
    t = shorten(clean_text(text))
    s = mc_stats(predict_logits([t], mc_dropout=True, passes=passes))
    conf, pf = s["conf"][0], s["prob"][0, 1]
    verdict = "UNCERTAIN — needs human review" if conf < tau else ("FAKE" if pf >= 0.5 else "REAL")
    exp = explainer.explain_instance(t, bert_proba, num_features=8, num_samples=lime_samples)
    print(f"Verdict   : {verdict}")
    print(f"P(fake)   : {pf:.3f}   (MC Dropout mean of {passes} passes, std {s['std_fake'][0]:.3f})")
    print(f"Confidence: {conf:.3f}   threshold tau = {tau:.3f}   mutual information = {s['mi'][0]:.5f}")
    print("Key words : " + ", ".join(f"{w} ({'FAKE' if v > 0 else 'REAL'} {abs(v):.2f})" for w, v in exp.as_list()))

analyse("The Senate voted 52-48 on Tuesday to approve the budget resolution, sending the measure to the House, "
        "where Republican leaders said they expected a vote later this week. Democrats criticised the plan, "
        "saying it would add to the deficit, while the White House said the president would sign it.")
print("-" * 80)
analyse("BREAKING: Doctors are FURIOUS after a mom discovered this one weird trick that cures diabetes overnight! "
        "The mainstream media won t tell you the truth, but thousands of patriots are sharing it. Share before it gets deleted!")

### How to read your results (for the report)

* **Raw vs clean baseline:** if accuracy barely changes after cleaning, the classes are still separable by *writing style* (Reuters wire style — "said on Tuesday", "U.S." — vs opinionated blog style). Say this honestly: the ISOT benchmark measures **source style**, not truthfulness.
* **Transformer vs baseline:** in-domain both are very high. The interesting comparison is **cross-domain** and **calibration**.
* **Calibration:** if softmax ECE > temperature-scaled ECE, the network was over-confident and one parameter fixed it. Compare MC Dropout honestly — it may or may not beat temperature scaling.
* **Selective prediction:** error-detection AUROC well above 0.5 and the fact that "accuracy (flagged only)" is much lower than "accuracy (confident only)" show that low confidence really does signal likely errors.
* **Domain shift:** expect a large accuracy drop on COVID-19. The key question is whether the model *knows* it is on unfamiliar ground: a higher UNCERTAIN rate and higher mutual information on COVID posts say yes; high confidence with low accuracy says the uncertainty does **not** transfer — both are valid, reportable findings.
* **LIME deletion test:** if deleting LIME's top words hurts confidence much more than random words, the explanations are faithful to the model.